In [1]:
import pandas as pd
import sqlite3

In [2]:
df = pd.read_excel("ШЧ_2_corr.xlsx")
conn = sqlite3.connect("transport.db")

df.to_sql("inspections", conn, if_exists="replace", index=False)
print(df.columns)

Index(['Unnamed: 0', 'Дата осмотра', 'Осмотр проводил', 'Станция, перегон',
       'Объекты, количество отступлений', 'Классификация (Группа)',
       'Классификация (Вид)', 'Примечание', 'Крайний срок', 'Статус',
       'Причина', 'Дата устранения'],
      dtype='str')


In [3]:
# сколько всего нарушений

query = """
SELECT COUNT(*) AS total
FROM inspections
"""

print(pd.read_sql(query, conn))
# 12112 как на слайде

   total
0  12112


In [4]:
# задание 1 записи с непустой причиной

query = """
SELECT *
FROM inspections
WHERE "Причина" IS NOT NULL
"""

print(pd.read_sql(query, conn))
# пусто, тоесть причина не заполнена ни у одной записи

Empty DataFrame
Columns: [Unnamed: 0, Дата осмотра, Осмотр проводил, Станция, перегон, Объекты, количество отступлений, Классификация (Группа), Классификация (Вид), Примечание, Крайний срок, Статус, Причина, Дата устранения]
Index: []


In [5]:
# задание 2 крайний срок после 15 апреля 2019

query = """
SELECT COUNT(*) AS cnt
FROM inspections
WHERE DATE("Крайний срок") > '2019-04-15'
"""

print(pd.read_sql(query, conn))
# 10702 записи

     cnt
0  10702


In [6]:
# задание 3 топ-7 станций по нарушениям со статусом ЗАКРЫТО

query = """
SELECT "Станция, перегон", COUNT(*) AS cnt
FROM inspections
WHERE "Статус" = 'ЗАКРЫТО'
GROUP BY "Станция, перегон"
ORDER BY cnt DESC
LIMIT 7
"""

print(pd.read_sql(query, conn))
# станция_0 сильно впереди всех, 673

  Станция, перегон  cnt
0        станция_0  673
1       станция_60  299
2       станция_17  217
3        станция_3  200
4       станция_75  160
5       станция_80  158
6       станция_82  148


In [7]:
# задание 4 топ-3 группы с пустой причиной

query = """
SELECT "Классификация (Группа)", COUNT(*) AS cnt
FROM inspections
WHERE "Причина" IS NULL
GROUP BY "Классификация (Группа)"
ORDER BY cnt DESC
LIMIT 3
"""

print(pd.read_sql(query, conn))
#причина пустая везде, по этому это просто топ групп вообще

  Классификация (Группа)   cnt
0              группа_32  2600
1               группа_1  2324
2               группа_4  1709


In [8]:
# задание 5 устранили раньше крайнего срока

query = """
SELECT COUNT(*) AS cnt
FROM inspections
WHERE DATE("Дата устранения") < DATE("Крайний срок")
"""

print(pd.read_sql(query, conn))
# 12111 из 12112, у одной нет даты устранения

     cnt
0  12111


In [9]:
# задание 6 топ-5 сотрудников на станция_0

query = """
SELECT "Осмотр проводил", COUNT(*) AS cnt
FROM inspections
WHERE "Станция, перегон" = 'станция_0'
GROUP BY "Осмотр проводил"
ORDER BY cnt DESC
LIMIT 5
"""

print(pd.read_sql(query, conn))

  Осмотр проводил  cnt
0     Сотрудник_1  473
1     Сотрудник_0  419
2     Сотрудник_7  251
3    Сотрудник_32  103
4    Сотрудник_30   80


In [10]:
# задание 7 нарушения за май 2019

query = """
SELECT COUNT(*) AS cnt
FROM inspections
WHERE "Дата осмотра" BETWEEN '2019-05-01' AND '2019-05-31'
"""

print(pd.read_sql(query, conn))
# 2015 за май

    cnt
0  2015


In [11]:
# задание 8 статус УСТРАНЕН но нет даты устранения

query = """
SELECT *
FROM inspections
WHERE "Статус" = 'УСТРАНЕН' AND "Дата устранения" IS NULL
"""

print(pd.read_sql(query, conn))
# таких нет, статусы видимо закрывают только с датой
#print(pd.read_sql('SELECT "Статус", COUNT(*) FROM inspections GROUP BY "Статус"', conn))

Empty DataFrame
Columns: [Unnamed: 0, Дата осмотра, Осмотр проводил, Станция, перегон, Объекты, количество отступлений, Классификация (Группа), Классификация (Вид), Примечание, Крайний срок, Статус, Причина, Дата устранения]
Index: []


In [12]:
# задание 9 топ-5 станций с самым долгим средним сроком устранения
# julianday переводит дату в число дней, их можно вычитать

query = """
SELECT "Станция, перегон",
       ROUND(AVG(julianday("Дата устранения") - julianday("Дата осмотра")), 1) AS sr_dney,
       COUNT(*) AS cnt
FROM inspections
WHERE "Дата устранения" IS NOT NULL
GROUP BY "Станция, перегон"
ORDER BY sr_dney DESC
LIMIT 5
"""

print(pd.read_sql(query, conn))
# станция_9 дольше всех, в среднем 62 дня, но там всего 24 нарушения

  Станция, перегон  sr_dney  cnt
0        станция_9     62.4   24
1       станция_14     51.2   56
2       станция_17     45.5  272
3       станция_32     45.4   66
4       станция_45     43.8   43
